In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata

# ============================================================
# CONFIG (Switch c) — Quantiacs version
# ============================================================
np.random.seed(40)

N = 50                 # random assets
K = 252                # risk lookback (1 year)
mu_lookback = 20       # lifestyle param
rebalance_every = 1    # lifestyle param (daily)
gamma = 0.5            # lifestyle param
tc_lambda = 0.00       # lifestyle param
w_max = 0.10           # lifestyle param
long_only = True

alpha = 0.90           # lifestyle param (EWMA decay)
shrink_delta = 0.10    # optional shrinkage for SampleCov
ridge = 1e-8           # numerical stability

# If you want to limit history (optional). None = use full data.
MAX_DAYS = None  # e.g. 1500 or None

# ============================================================
# Load Quantiacs data + pick random liquid assets
# ============================================================
data = qndata.stocks.load_data()  # xarray DataArray(field, time, asset)
close = data.sel(field="close")
is_liquid = data.sel(field="is_liquid")

# choose assets that are liquid at the last date and have prices
liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
close_last = close.isel(time=-1)
good_assets = close.asset.values[(liquid_last.values) & np.isfinite(close_last.values)]

if len(good_assets) < N:
    raise ValueError(f"Not enough liquid assets available ({len(good_assets)}) to sample N={N}.")

assets = np.random.choice(good_assets, size=N, replace=False)

# extract close for chosen assets
close_sel = close.sel(asset=assets)

# Convert to pandas DataFrame: rows=time, cols=asset
close_df = close_sel.to_pandas()

# optional history truncation
if MAX_DAYS is not None and len(close_df) > MAX_DAYS:
    close_df = close_df.iloc[-MAX_DAYS:]

# compute simple returns
rets = close_df.pct_change().replace([np.inf, -np.inf], np.nan)

# basic cleaning: forward fill small gaps, then fill remaining with 0
rets = rets.fillna(method="ffill").fillna(0.0)

# ============================================================
# HELPERS: PSD projection + covariance estimators
# ============================================================
def make_psd(S, eps=1e-10):
    S = 0.5 * (S + S.T)
    vals, vecs = np.linalg.eigh(S)
    vals_clipped = np.maximum(vals, eps)
    return (vecs * vals_clipped) @ vecs.T

def sample_cov_shrink(returns_window, delta=0.0):
    S = np.cov(returns_window, rowvar=False, ddof=1)
    S = 0.5 * (S + S.T)
    if delta > 0:
        D = np.diag(np.diag(S))
        S = (1 - delta) * S + delta * D
    return make_psd(S, eps=1e-12)

def ewma_cov_update(r_t, S_prev, alpha=0.90):
    r_t = r_t.reshape(-1, 1)
    S = (1 - alpha) * (r_t @ r_t.T) + alpha * S_prev
    return make_psd(S, eps=1e-12)

# ============================================================
# CVXPY SOLVER (same as your working baseline)
# maximize mu^T w - gamma * w^T Sigma w - tc_lambda * ||w - w_prev||_1
# ============================================================
def solve_weights(mu, Sigma, w_prev,
                  gamma=0.5, tc_lambda=0.0,
                  long_only=True, w_max=0.10):
    n = len(mu)
    w = cp.Variable(n)

    Sigma = 0.5 * (Sigma + Sigma.T) + ridge * np.eye(n)
    Sigma = make_psd(Sigma, eps=1e-12)

    obj = mu @ w - gamma * cp.quad_form(w, Sigma)

    if tc_lambda > 0:
        obj -= tc_lambda * cp.norm1(w - w_prev)

    constraints = [cp.sum(w) == 1]
    if long_only:
        constraints += [w >= 0]
    if w_max is not None:
        constraints += [w <= w_max]

    prob = cp.Problem(cp.Maximize(obj), constraints)
    prob.solve(solver=cp.OSQP, verbose=False)

    if w.value is None:
        return np.ones(n) / n
    return np.array(w.value).reshape(-1)

# ============================================================
# BACKTEST: compare SampleCov vs EWMA (same structure)
# ============================================================
def run_backtest(returns_df, method="sample"):
    T, N = returns_df.shape
    w = np.ones(N) / N
    w_hist = np.zeros((T, N))
    port_ret = np.zeros(T)
    risk_trace = np.zeros(T)

    # init EWMA covariance using early sample cov
    init_win = min(K, max(50, mu_lookback + 1))
    S_ewma = sample_cov_shrink(returns_df.iloc[:init_win].values, delta=shrink_delta)

    for t in range(T):
        r_t = returns_df.iloc[t].values
        port_ret[t] = w @ r_t
        w_hist[t] = w

        if (t % rebalance_every == 0) and (t >= max(K, mu_lookback) + 1):
            mu = returns_df.iloc[t - mu_lookback:t].mean().values

            if method == "sample":
                win = returns_df.iloc[t - K:t].values
                Sigma = sample_cov_shrink(win, delta=shrink_delta)
            elif method == "ewma":
                Sigma = S_ewma.copy()
            else:
                raise ValueError("method must be 'sample' or 'ewma'")

            w_new = solve_weights(
                mu=mu,
                Sigma=Sigma,
                w_prev=w,
                gamma=gamma,
                tc_lambda=tc_lambda,
                long_only=long_only,
                w_max=w_max
            )

            risk_trace[t] = float(w_new.T @ Sigma @ w_new)
            w = w_new

        if t >= 1:
            S_ewma = ewma_cov_update(r_t, S_ewma, alpha=alpha)

    equity = (1 + port_ret).cumprod()
    return equity, port_ret, w_hist, risk_trace

eq_sample, pr_sample, w_sample, risk_sample = run_backtest(rets, method="sample")
eq_ewma, pr_ewma, w_ewma, risk_ewma = run_backtest(rets, method="ewma")

# ============================================================
# REPORT + PLOTS (same as your baseline)
# ============================================================
def sharpe(x):
    x = np.asarray(x)
    if x.std() < 1e-12:
        return 0.0
    return np.sqrt(252) * x.mean() / x.std()

def max_drawdown(equity):
    equity = np.asarray(equity)
    peak = np.maximum.accumulate(equity)
    dd = equity / peak - 1.0
    return dd.min()

print("=== RESULTS (Quantiacs stocks, random 50) ===")
print(f"SampleCov: Sharpe={sharpe(pr_sample):.3f}  MaxDD={max_drawdown(eq_sample):.3%}")
print(f"EWMA     : Sharpe={sharpe(pr_ewma):.3f}  MaxDD={max_drawdown(eq_ewma):.3%}")

plt.figure()
plt.plot(eq_sample, label="SampleCov (rolling)")
plt.plot(eq_ewma, label="EWMA (time-varying)")
plt.title("Equity curve")
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
plt.plot(pd.Series(risk_sample).rolling(10).mean(), label="Risk proxy (SampleCov)")
plt.plot(pd.Series(risk_ewma).rolling(10).mean(), label="Risk proxy (EWMA)")
plt.title("Approx risk (w^T Sigma w) at rebalance days (smoothed)")
plt.legend()
plt.grid(True)
plt.show()

plt.figure()
plt.plot(np.abs(np.diff(w_sample, axis=0)).sum(axis=1), label="Turnover (SampleCov)")
plt.plot(np.abs(np.diff(w_ewma, axis=0)).sum(axis=1), label="Turnover (EWMA)")
plt.title("Daily turnover proxy: sum |w_t - w_{t-1}|")
plt.legend()
plt.grid(True)
plt.show()

win = 63
vol_s = pd.Series(pr_sample).rolling(win).std() * np.sqrt(252)
vol_e = pd.Series(pr_ewma).rolling(win).std() * np.sqrt(252)

plt.figure()
plt.plot(vol_s, label="Rolling vol (SampleCov)")
plt.plot(vol_e, label="Rolling vol (EWMA)")
plt.title(f"Rolling annualized volatility (window={win})")
plt.legend()
plt.grid(True)
plt.show()

w_diff_l1 = np.abs(w_ewma - w_sample).sum(axis=1)

plt.figure()
plt.plot(w_diff_l1)
plt.title("Portfolio difference: L1 norm |w_EWMA - w_Sample|")
plt.grid(True)
plt.show()

Kshow = 15
avg_w = w_sample.mean(axis=0)
top_idx = np.argsort(-avg_w)[:Kshow]

plt.figure(figsize=(10, 4))
plt.imshow(w_sample[:, top_idx].T, aspect="auto", interpolation="nearest")
plt.title(f"SampleCov weights heatmap (top {Kshow} assets)")
plt.ylabel("Asset (top by avg weight)")
plt.xlabel("Time")
plt.colorbar(label="weight")
plt.show()

plt.figure(figsize=(10, 4))
plt.imshow(w_ewma[:, top_idx].T, aspect="auto", interpolation="nearest")
plt.title(f"EWMA weights heatmap (top {Kshow} assets)")
plt.ylabel("Asset (top by avg weight)")
plt.xlabel("Time")
plt.colorbar(label="weight")
plt.show()

print("Random assets used:", list(rets.columns))